<a href="https://colab.research.google.com/github/herrrickshaw/herrrickshaw/blob/main/SEBI_web_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import time
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service # Import Service
from webdriver_manager.chrome import ChromeDriverManager # Import ChromeDriverManager

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

def run_sebi_scraper():
    # =====================================================================
    # 2. HEADLESS CHROME SETTINGS FOR GOOGLE COLAB
    # =====================================================================
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--no-default-browser-check') # Additional argument
    chrome_options.add_argument('--ignore-ssl-errors') # Additional argument
    chrome_options.add_argument('--ignore-certificate-errors') # Additional argument
    chrome_options.add_argument('--disable-extensions') # Additional argument
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    chrome_options.binary_location = '/usr/bin/google-chrome' # Corrected path to Google Chrome binary

    # SEBI URL targeting Public Issues -> Draft Offer Documents
    target_url = "https://www.sebi.gov.in/sebiweb/home/HomeAction.do?doListing=yes&sid=3&ssid=15&smid=10"

    print("Initializing headless browser in Colab...")
    # Use ChromeDriverManager to automatically download and install the correct chromedriver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.get(target_url)

    extracted_records = []

    print("Connected to SEBI database. Parsing public listings...")

    page_num = 1
    page_limit = 40 # Set the limit to 40 pages

    print("\nProceeding to scrape without year-specific filtering (default view)...")

    while True:
        if page_num > page_limit:
            print(f"  Page limit of {page_limit} reached.")
            break

        print(f"  Processing Page {page_num}...")

        # Wait until dynamic content table container resolves
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//table | //div[contains(@class, 'card-table')]"))
            )
        except Exception:
            print("  Pagination timeline or table timeout encountered. Finalizing collection.")
            break

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # SEBI lists these documents in conventional <tr> rows inside table architectures
        table_rows = soup.find_all('tr')

        if not table_rows:
            # Fallback if structural card views are served dynamically
            table_rows = soup.find_all('div', class_='row-cards')

        for row in table_rows:
            cells = row.find_all('td')
            if not cells:
                continue

            # Usually date lives in the first or second column; look for standard dates (DD MMM YYYY / DD/MM/YYYY)
            date_text = cells[0].get_text(strip=True)

            # Locate anchor tag containing document link and string title
            anchor = row.find('a')
            if anchor:
                title = anchor.get_text(strip=True)
                href = anchor.get('href', '')

                # Rigid regulatory target criteria: Filter for DRHPs only
                if "DRHP" in title.upper() or "DRAFT RED HERRING" in title.upper():
                    # Construct absolute safe URLs from relative paths
                    full_url = href if href.startswith('http') else f"https://www.sebi.gov.in{href}"
                    extracted_records.append([date_text, title, full_url])

        # Debugging Pagination Interactions
        pagination_xpath = "//a[contains(text(), 'Next') or contains(@class, 'next') or contains(@onclick, 'next')]"
        next_buttons = driver.find_elements(By.XPATH, pagination_xpath)

        print(f"  DEBUG: Found {len(next_buttons)} elements matching pagination XPath: {pagination_xpath}")
        for i, btn in enumerate(next_buttons):
            try:
                print(f"    DEBUG: Button {i+1} Text: '{btn.text}', Class: '{btn.get_attribute('class')}', Displayed: {btn.is_displayed()}, Enabled: {btn.is_enabled()}")
            except Exception as debug_e:
                print(f"    DEBUG: Could not get attributes for button {i+1}: {debug_e}")

        try:
            # Wait for the next button to be clickable
            next_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, pagination_xpath))
            )

            # Some sites might apply a 'disabled' class even if the element is technically clickable.
            # Check for this as an explicit end-of-pagination indicator.
            btn_classes = next_btn.get_attribute("class")
            if btn_classes and "disabled" in btn_classes:
                print(f"  Next button found but is disabled, indicating end of pagination.")
                break

            print(f"  DEBUG: Successfully found clickable next button with text: '{next_btn.text}' and class: '{btn_classes}'")
            driver.execute_script("arguments[0].click();", next_btn)
            page_num += 1
            time.sleep(2)  # Defensive throttling delay to maintain clean server sessions
        except Exception as e:
            # This catch will handle TimeoutException if the next button is not found or not clickable
            # or any other exception during the click operation.
            print(f"  Could not find or click the next page button, or an error occurred: {e}. Assuming end of pagination.")
            break

    driver.quit()

    # =====================================================================
    # 3. GENERATE STYLIZED EXCEL OUTPUT (USING OPENPYXL)
    # =====================================================================
    print(f"\nExtraction complete. Total matching DRHP filings harvested: {len(extracted_records)}")
    print("Formatting and building Excel spreadsheet...")

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "SEBI DRHP Filings"

    # Enable explicit visual layout grids
    ws.views.sheetView[0].showGridLines = True

    # Design elements palette (Steel Blue theme)
    header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    zebra_fill = PatternFill(start_color="F2F6F9", end_color="F2F6F9", fill_type="solid")
    header_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Segoe UI", size=10, bold=False, color="000000")
    link_font = Font(name="Segoe UI", size=10, bold=False, color="0563C1", underline="single")

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    # Add table column headers
    headers = ["Filing Date", "Company / Prospectus Document Title", "Direct SEBI Document Link"]
    ws.append(headers)

    # Apply styling rules to the header row
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center" if col_idx == 1 else "left", vertical="center")
        cell.border = thin_border

    # Inject dataset and apply alternating row tints and structural alignments
    for row_idx, data_row in enumerate(extracted_records, 2):
        ws.append(data_row)

        # Date Styling
        date_cell = ws.cell(row=row_idx, column=1)
        date_cell.alignment = Alignment(horizontal="center", vertical="center")

        # Title Styling
        title_cell = ws.cell(row=row_idx, column=2)
        title_cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)

        # Hyperlink Destination Mapping
        link_cell = ws.cell(row=row_idx, column=3)
        link_cell.hyperlink = data_row[2]
        link_cell.font = link_font
        link_cell.alignment = Alignment(horizontal="left", vertical="center")

        # Loop styling attributes across active matrix row
        for col_idx in range(1, 4):
            c = ws.cell(row=row_idx, column=col_idx)
            if col_idx != 3:
                c.font = data_font
            c.border = thin_border
            if row_idx % 2 == 0:
                c.fill = zebra_fill

    # Freeze header top pane row
    ws.freeze_panes = 'A2'

    # Set intentional, readable width distribution for parameters
    ws.column_dimensions['A'].width = 16   # Balanced Date Width
    ws.column_dimensions['B'].width = 65   # Broad text buffer for Company Names
    ws.column_dimensions['C'].width = 50   # Compact containment for URLs

    # Save active workbook down to local Colab filesystem virtual path
    output_filename = "SEBI_DRHP_Filings_Default_View.xlsx"
    wb.save(output_filename)
    print(f"Spreadsheet generated completely and saved as '{output_filename}'!")

if __name__ == "__main__":
    run_sebi_scraper()

Initializing headless browser in Colab...
Connected to SEBI database. Parsing public listings...

Proceeding to scrape without year-specific filtering (default view)...
  Processing Page 1...
  DEBUG: Found 2 elements matching pagination XPath: //a[contains(text(), 'Next') or contains(@class, 'next') or contains(@onclick, 'next')]
    DEBUG: Button 1 Text: 'Next ', Class: 'None', Displayed: True, Enabled: True
    DEBUG: Button 2 Text: 'Next ', Class: 'None', Displayed: True, Enabled: True
  DEBUG: Successfully found clickable next button with text: 'Next ' and class: 'None'
  Processing Page 2...
  DEBUG: Found 2 elements matching pagination XPath: //a[contains(text(), 'Next') or contains(@class, 'next') or contains(@onclick, 'next')]
    DEBUG: Button 1 Text: 'Next ', Class: 'None', Displayed: True, Enabled: True
    DEBUG: Button 2 Text: 'Next ', Class: 'None', Displayed: True, Enabled: True
  DEBUG: Successfully found clickable next button with text: 'Next ' and class: 'None'

In [20]:
import pandas as pd

# Load the Excel file into a pandas DataFrame
excel_file_path = 'SEBI_DRHP_Filings_Default_View.xlsx'
df_scraped_data = pd.read_excel(excel_file_path)

# Display the first few rows of the DataFrame
display(df_scraped_data.head())

,Filing Date,Company / Prospectus Document Title,Direct SEBI Document Link
0,"Jun 11, 2026",Caliber Mining and Logistics Limited - Corrige...,https://www.sebi.gov.in/filings/public-issues/...
1,"Jun 10, 2026",Aastha Spintex Limited - Addendum to DRHP,https://www.sebi.gov.in/filings/public-issues/...
2,"Jun 09, 2026",Zepto Limited - UDRHP1,https://www.sebi.gov.in/filings/public-issues/...
3,"Jun 03, 2026",Laser Power and Infra Limited - Corrigendum to...,https://www.sebi.gov.in/filings/public-issues/...
4,"Jun 02, 2026",Matangi Rubber Limited - DRHP,https://www.sebi.gov.in/filings/public-issues/...


### Selecting Options from a Dropdown with Selenium

Selenium provides a `Select` class specifically for interacting with dropdowns (`<select>` HTML elements). You first need to locate the dropdown element and then pass it to the `Select` constructor. After that, you can use methods like `select_by_visible_text()`, `select_by_value()`, or `select_by_index()` to choose an option.

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- Setup (similar to your scraper) ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.binary_location = '/usr/bin/google-chrome'

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# For demonstration, let's assume a simple HTML page with a dropdown
# In a real scenario, you would navigate to your target_url
# driver.get("your_target_url_with_dropdown")

# Create a dummy HTML page with a dropdown for demonstration
dummy_html = """
<html>
<body>
    <select id='myDropdown'>
        <option value='option1'>Option One</option>
        <option value='option2'>Option Two</option>
        <option value='option3'>Option Three</option>
    </select>
</body>
</html>
"""

driver.get("data:text/html;charset=utf-8," + dummy_html)

print("Demonstrating dropdown selection:")

try:
    # 1. Locate the dropdown element
    dropdown_element = driver.find_element(By.ID, 'myDropdown')

    # 2. Create a Select object
    select = Select(dropdown_element)

    # 3. Select an option by its visible text
    print("Selecting 'Option Two' by visible text...")
    select.select_by_visible_text('Option Two')
    print(f"Currently selected: {select.first_selected_option.text}")
    time.sleep(1)

    # 4. Select an option by its value attribute
    print("Selecting 'Option One' by value...")
    select.select_by_value('option1')
    print(f"Currently selected: {select.first_selected_option.text}")
    time.sleep(1)

    # 5. Select an option by its index (0-based)
    print("Selecting 'Option Three' by index (index 2)...")
    select.select_by_index(2)
    print(f"Currently selected: {select.first_selected_option.text}")
    time.sleep(1)

except Exception as e:
    print(f"An error occurred during dropdown interaction: {e}")
finally:
    driver.quit()


Demonstrating dropdown selection:
Selecting 'Option Two' by visible text...
Currently selected: Option Two
Selecting 'Option One' by value...
Currently selected: Option One
Selecting 'Option Three' by index (index 2)...
Currently selected: Option Three
